# VisionInspect — 04 Specialist Router (MVTec per-category)

The previous router used `MVTec_AD` as one class. That does not match the deployment architecture because PatchCore is trained as **15 independent MVTec specialists**.

This router therefore has **18 specialist destinations**:

- `DAGM`
- `KolektorSDD2`
- `Magnetic_Tile`
- 15 MVTec AD categories, each mapped to its own PatchCore model.

The router identifies **which trained specialist should receive the image**; it does not perform defect classification.


In [3]:
from pathlib import Path
import random
import shutil
import json
import time

import pandas as pd
import yaml
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from ultralytics import YOLO

SEED = 42
random.seed(SEED)

DATA_ROOT = Path("datasets")
ROUTER_ROOT = DATA_ROOT / "domain_router_v2"

SOURCE_ROOTS = {
    "DAGM": DATA_ROOT / "yolo_segmentation" / "DAGM",
    "KolektorSDD2": DATA_ROOT / "yolo_segmentation" / "KolektorSDD2",
    "Magnetic_Tile": DATA_ROOT / "yolo_segmentation" / "Magnetic_Tile",
    "MVTec_AD": DATA_ROOT / "MVTec_AD",
}

MVTEC_CLASSES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper",
]

ROUTER_CLASSES = [
    "DAGM", "KolektorSDD2", "Magnetic_Tile"
] + [f"MVTec_{name}" for name in MVTEC_CLASSES]

DOMAIN_TO_ID = {name: i for i, name in enumerate(ROUTER_CLASSES)}
ID_TO_DOMAIN = {i: name for name, i in DOMAIN_TO_ID.items()}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

MODEL_NAME = "yolo26n-cls.pt"
IMAGE_SIZE = 224
EPOCHS = 75
BATCH_SIZE = 32
WORKERS = 4
DEVICE = None

print(f"Router classes: {len(ROUTER_CLASSES)}")
print(ROUTER_CLASSES)


Router classes: 18
['DAGM', 'KolektorSDD2', 'Magnetic_Tile', 'MVTec_bottle', 'MVTec_cable', 'MVTec_capsule', 'MVTec_carpet', 'MVTec_grid', 'MVTec_hazelnut', 'MVTec_leather', 'MVTec_metal_nut', 'MVTec_pill', 'MVTec_screw', 'MVTec_tile', 'MVTec_toothbrush', 'MVTec_transistor', 'MVTec_wood', 'MVTec_zipper']


## 1. Image discovery — masks excluded

In [4]:
def find_images(root, exclude_masks=True):
    root = Path(root)
    if not root.exists():
        return []

    images = []
    mask_suffixes = ("_mask", "-mask", "_gt", "-gt", "_label", "-label")
    label_directories = {
        "mask", "masks", "label", "labels", "gt",
        "ground_truth", "ground-truth"
    }

    for path in root.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        if exclude_masks:
            if path.stem.lower().endswith(mask_suffixes):
                continue
            if any(part.lower() in label_directories for part in path.parts):
                continue

        images.append(path)

    return sorted(images)


def find_prepared_images(root):
    root = Path(root)
    images = []

    for split in ["train", "val", "test"]:
        directory = root / "images" / split
        if directory.exists():
            images.extend(find_images(directory, exclude_masks=True))

    return sorted(images)


## 2. MVTec category discovery

In [5]:
def find_mvtec_category_images(root, categories):
    root = Path(root)
    records = []

    for category in categories:
        category_root = root / category

        if not category_root.exists():
            print(f"WARNING: missing MVTec category: {category_root}")
            continue

        for image_path in find_images(category_root, exclude_masks=True):
            records.append({
                "router_class": f"MVTec_{category}",
                "source_dataset": "MVTec_AD",
                "mvtec_category": category,
                "source_path": str(image_path),
            })

    return records


## 3. Build specialist-router inventory

In [6]:
records = []

# Dataset-level specialists
for domain in ["DAGM", "KolektorSDD2", "Magnetic_Tile"]:
    for image_path in find_prepared_images(SOURCE_ROOTS[domain]):
        records.append({
            "router_class": domain,
            "source_dataset": domain,
            "mvtec_category": None,
            "source_path": str(image_path),
        })

# Individual MVTec PatchCore specialists
records.extend(
    find_mvtec_category_images(
        SOURCE_ROOTS["MVTec_AD"],
        MVTEC_CLASSES,
    )
)

inventory = pd.DataFrame(records)

if inventory.empty:
    raise RuntimeError("No images found. Check SOURCE_ROOTS.")

print(f"Total router images: {len(inventory):,}")
display(
    inventory.groupby("router_class")
    .size()
    .rename("images")
    .to_frame()
)


Total router images: 13,820


,images
router_class,
DAGM,3786
KolektorSDD2,3336
MVTec_bottle,292
MVTec_cable,374
MVTec_capsule,351
MVTec_carpet,397
MVTec_grid,342
MVTec_hazelnut,501
MVTec_leather,369


## 4. Validate all 15 MVTec categories

In [7]:
found_mvtec = sorted(
    inventory.loc[
        inventory["source_dataset"] == "MVTec_AD",
        "mvtec_category"
    ].dropna().unique()
)

missing_mvtec = sorted(set(MVTEC_CLASSES) - set(found_mvtec))

print("Expected:", len(MVTEC_CLASSES))
print("Found:   ", len(found_mvtec))

if missing_mvtec:
    print("Missing:", missing_mvtec)

assert not missing_mvtec, "Missing one or more expected MVTec categories."


Expected: 15
Found:    15


## 5. Inspect class imbalance before training

In [8]:
class_counts = (
    inventory.groupby("router_class")
    .size()
    .sort_values(ascending=False)
    .rename("images")
    .to_frame()
)

display(class_counts)

print("Largest:", class_counts.images.max())
print("Smallest:", class_counts.images.min())
print("Imbalance ratio:", round(class_counts.images.max() / class_counts.images.min(), 2))


,images
router_class,
DAGM,3786
KolektorSDD2,3336
Magnetic_Tile,1344
MVTec_hazelnut,501
MVTec_screw,480
MVTec_pill,434
MVTec_carpet,397
MVTec_zipper,391
MVTec_cable,374


Largest: 3786
Smallest: 102
Imbalance ratio: 37.12


## 6. Stratified 70/15/15 split

The router split is independent of the specialist models' training split.

For MVTec, both normal and defective images may be used for router training because the router learns **category identity**, while PatchCore itself remains trained only on the normal training images.


In [9]:
train_df, temp_df = train_test_split(
    inventory,
    test_size=0.30,
    stratify=inventory["router_class"],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["router_class"],
    random_state=SEED,
)

train_df = train_df.assign(split="train")
val_df = val_df.assign(split="val")
test_df = test_df.assign(split="test")

router_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

display(
    router_df.groupby(["split", "router_class"])
    .size()
    .unstack(fill_value=0)
)


router_class,DAGM,KolektorSDD2,MVTec_bottle,MVTec_cable,MVTec_capsule,MVTec_carpet,MVTec_grid,MVTec_hazelnut,MVTec_leather,MVTec_metal_nut,MVTec_pill,MVTec_screw,MVTec_tile,MVTec_toothbrush,MVTec_transistor,MVTec_wood,MVTec_zipper,Magnetic_Tile
split,,,,,,,,,,,,,,,,,,
test,568,500,44,56,52,60,52,75,56,50,65,72,52,15,47,49,59,201
train,2650,2335,204,262,246,278,239,351,258,235,304,336,243,71,219,228,274,941
val,568,501,44,56,53,59,51,75,55,50,65,72,52,16,47,49,58,202


## 7. Build YOLO classification dataset

In [8]:
def build_router_dataset(dataframe, output_root, overwrite=False):
    output_root = Path(output_root)

    if overwrite and output_root.exists():
        shutil.rmtree(output_root)

    for split in ["train", "val", "test"]:
        for router_class in ROUTER_CLASSES:
            (output_root / split / router_class).mkdir(
                parents=True,
                exist_ok=True,
            )

    manifest = []

    for index, row in dataframe.reset_index(drop=True).iterrows():
        source = Path(row["source_path"])

        destination = (
            output_root
            / row["split"]
            / row["router_class"]
            / f"{index:08d}_{source.name}"
        )

        if not destination.exists():
            shutil.copy2(source, destination)

        manifest.append({
            "index": index,
            "split": row["split"],
            "router_class": row["router_class"],
            "source_dataset": row["source_dataset"],
            "mvtec_category": row["mvtec_category"],
            "source_path": str(source),
            "router_path": str(destination),
        })

    return pd.DataFrame(manifest)


manifest = build_router_dataset(
    router_df,
    ROUTER_ROOT,
    overwrite=False,
)

manifest.to_csv(ROUTER_ROOT / "manifest.csv", index=False)
print("Router dataset:", ROUTER_ROOT.resolve())


Router dataset: /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router_v2


## 8. Validate generated dataset and check for mask leakage

In [9]:
def count_generated_images(root):
    rows = []

    for split in ["train", "val", "test"]:
        for router_class in ROUTER_CLASSES:
            directory = Path(root) / split / router_class
            rows.append({
                "split": split,
                "router_class": router_class,
                "images": len(find_images(directory, exclude_masks=True)),
            })

    return pd.DataFrame(rows)


generated_counts = count_generated_images(ROUTER_ROOT)

display(
    generated_counts.pivot(
        index="router_class",
        columns="split",
        values="images",
    ).fillna(0).astype(int)
)

suspicious = []

for path in ROUTER_ROOT.rglob("*"):
    if not path.is_file():
        continue
    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue

    if path.stem.lower().endswith(
        ("_mask", "-mask", "_gt", "-gt", "_label", "-label")
    ):
        suspicious.append(path)

print(f"Potential mask files: {len(suspicious)}")

assert not suspicious, "Mask/ground-truth files were included in router dataset."


split,test,train,val
router_class,,,
DAGM,568,2650,568
KolektorSDD2,500,2335,501
MVTec_bottle,44,204,44
MVTec_cable,56,262,56
MVTec_capsule,52,246,53
MVTec_carpet,60,278,59
MVTec_grid,52,239,51
MVTec_hazelnut,75,351,75
MVTec_leather,56,258,55


Potential mask files: 0


## 9. Dataset metadata

In [10]:
router_yaml = {
    "path": str(ROUTER_ROOT),
    "train": "train",
    "val": "val",
    "test": "test",
    "nc": len(ROUTER_CLASSES),
    "names": ROUTER_CLASSES,
}

yaml_path = ROUTER_ROOT / "dataset.yaml"
yaml_path.write_text(
    yaml.safe_dump(router_yaml, sort_keys=False),
    encoding="utf-8",
)

print(yaml_path.read_text())


path: datasets/domain_router_v2
train: train
val: val
test: test
nc: 18
names:
- DAGM
- KolektorSDD2
- Magnetic_Tile
- MVTec_bottle
- MVTec_cable
- MVTec_capsule
- MVTec_carpet
- MVTec_grid
- MVTec_hazelnut
- MVTec_leather
- MVTec_metal_nut
- MVTec_pill
- MVTec_screw
- MVTec_tile
- MVTec_toothbrush
- MVTec_transistor
- MVTec_wood
- MVTec_zipper



## 10. Train YOLO26n specialist router

In [11]:
model = YOLO(MODEL_NAME)

start_time = time.perf_counter()

model.train(
    data=str(ROUTER_ROOT),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    seed=SEED,
    pretrained=True,
    patience=15,
    project="runs/classify",
    name="specialist_router_v2",
    exist_ok=True,
    plots=True,
)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time / 60:.2f} minutes")
print("Best model:", model.trainer.best)


New https://pypi.org/project/ultralytics/8.4.120 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 7708MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/domain_router_v2, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=75, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mi

/home/zubair/AI Development Directory/anomaly-detection/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train: /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router_v2/train... found 9674 images in 18 classes ✅ 
val: /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router_v2/val... found 2073 images in 18 classes ✅ 
test: /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router_v2/test... found 2073 images in 18 classes ✅ 
Overriding model.yaml nc=1000 with nc=18

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]            

## 11. Validation

In [15]:
# load best model from training

path_to_best_model = "/home/zubair/AI Development Directory/anomaly-detection/runs/classify/runs/classify/specialist_router_v2/weights/best.pt"

model = YOLO(path_to_best_model)
val_results = model.val(
    data=str(ROUTER_ROOT),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
)

print("Top-1:", getattr(val_results, "top1", None))
print("Top-5:", getattr(val_results, "top5", None))


Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 7708MiB)
YOLO26n-cls summary (fused): 47 layers, 1,549,082 parameters, 0 gradients, 3.2 GFLOPs
train: /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router_v2/train... found 9674 images in 18 classes ✅ 
val: /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router_v2/val... found 2073 images in 18 classes ✅ 
test: /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router_v2/test... found 2073 images in 18 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5752.4±2169.5 MB/s, size: 191.4 KB)
val: Scanning /home/zubair/AI Development Directory/anomaly-detection/datasets/domain_router_v2/val... 2073 images, 0 corrupt: 100% ━━━━━━━━━━━━ 2073/2073 414.0Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 65/65 20.8it/s 3.1s0.2s
                   all          1          1
Speed: 0.1ms preprocess

## 12. Test-set routing evaluation

In [16]:
test_rows = []

for true_class in ROUTER_CLASSES:
    test_directory = ROUTER_ROOT / "test" / true_class

    for image_path in find_images(test_directory, exclude_masks=True):
        result = model.predict(
            source=str(image_path),
            imgsz=IMAGE_SIZE,
            device=DEVICE,
            verbose=False,
        )[0]

        predicted_id = int(result.probs.top1)
        confidence = float(result.probs.top1conf)
        predicted_class = ID_TO_DOMAIN[predicted_id]

        test_rows.append({
            "image": str(image_path),
            "true_class": true_class,
            "pred_class": predicted_class,
            "confidence": confidence,
            "correct": true_class == predicted_class,
        })

test_predictions = pd.DataFrame(test_rows)

y_true = test_predictions["true_class"]
y_pred = test_predictions["pred_class"]

print(
    classification_report(
        y_true,
        y_pred,
        labels=ROUTER_CLASSES,
        target_names=ROUTER_CLASSES,
        digits=4,
        zero_division=0,
    )
)


                  precision    recall  f1-score   support

            DAGM     1.0000    1.0000    1.0000       568
    KolektorSDD2     1.0000    1.0000    1.0000       500
   Magnetic_Tile     0.0000    0.0000    0.0000       201
    MVTec_bottle     0.0000    0.0000    0.0000        44
     MVTec_cable     0.0000    0.0000    0.0000        56
   MVTec_capsule     0.0000    0.0000    0.0000        52
    MVTec_carpet     0.0000    0.0000    0.0000        60
      MVTec_grid     0.0000    0.0000    0.0000        52
  MVTec_hazelnut     0.0000    0.0000    0.0000        75
   MVTec_leather     0.0000    0.0000    0.0000        56
 MVTec_metal_nut     0.0000    0.0000    0.0000        50
      MVTec_pill     0.0000    0.0000    0.0000        65
     MVTec_screw     0.0000    0.0000    0.0000        72
      MVTec_tile     0.0000    0.0000    0.0000        52
MVTec_toothbrush     0.0000    0.0000    0.0000        15
MVTec_transistor     0.0000    0.0000    0.0000        47
      MVTec_w

## 13. Specialist-level metrics

In [17]:
report = pd.DataFrame(
    classification_report(
        y_true,
        y_pred,
        labels=ROUTER_CLASSES,
        target_names=ROUTER_CLASSES,
        output_dict=True,
        zero_division=0,
    )
).T

display(
    report.loc[
        ROUTER_CLASSES,
        ["precision", "recall", "f1-score", "support"],
    ].sort_values("recall")
)


,precision,recall,f1-score,support
MVTec_bottle,0.0,0.0,0.0,44.0
Magnetic_Tile,0.0,0.0,0.0,201.0
MVTec_capsule,0.0,0.0,0.0,52.0
MVTec_cable,0.0,0.0,0.0,56.0
MVTec_carpet,0.0,0.0,0.0,60.0
MVTec_grid,0.0,0.0,0.0,52.0
MVTec_leather,0.0,0.0,0.0,56.0
MVTec_hazelnut,0.0,0.0,0.0,75.0
MVTec_screw,0.0,0.0,0.0,72.0
MVTec_tile,0.0,0.0,0.0,52.0


## 14. Prediction distribution and confusion matrix

In [18]:
display(
    test_predictions["pred_class"]
    .value_counts()
    .reindex(ROUTER_CLASSES, fill_value=0)
    .rename("predictions")
    .to_frame()
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=ROUTER_CLASSES,
)

fig, ax = plt.subplots(figsize=(16, 14))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=ROUTER_CLASSES,
).plot(
    ax=ax,
    xticks_rotation=90,
    values_format="d",
)

ax.set_title("VisionInspect Specialist Router — Test")
plt.tight_layout()
plt.show()


,predictions
pred_class,
DAGM,568
KolektorSDD2,500
Magnetic_Tile,44
MVTec_bottle,56
MVTec_cable,52
MVTec_capsule,60
MVTec_carpet,52
MVTec_grid,75
MVTec_hazelnut,56


<Figure size 1600x1400 with 2 Axes>

## 15. Confidence and reject analysis

In [19]:
confidence_summary = (
    test_predictions
    .groupby("true_class")["confidence"]
    .agg(["count", "mean", "median", "min", "max"])
)

display(confidence_summary)

threshold_rows = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]:
    accepted = test_predictions["confidence"] >= threshold

    threshold_rows.append({
        "threshold": threshold,
        "accepted_accuracy": (
            test_predictions.loc[accepted, "correct"].mean()
            if accepted.any() else None
        ),
        "rejection_rate": 1.0 - accepted.mean(),
        "accepted_samples": int(accepted.sum()),
    })

display(pd.DataFrame(threshold_rows))


,count,mean,median,min,max
true_class,,,,,
DAGM,568,0.999985,1.000000,0.994384,1.000000
KolektorSDD2,500,1.000000,1.000000,0.999980,1.000000
MVTec_bottle,44,1.000000,1.000000,0.999990,1.000000
MVTec_cable,56,1.000000,1.000000,1.000000,1.000000
MVTec_capsule,52,1.000000,1.000000,1.000000,1.000000
MVTec_carpet,60,1.000000,1.000000,1.000000,1.000000
MVTec_grid,52,0.999529,0.999998,0.992690,1.000000
MVTec_hazelnut,75,1.000000,1.000000,1.000000,1.000000
MVTec_leather,56,0.997991,0.998030,0.996827,0.998734


,threshold,accepted_accuracy,rejection_rate,accepted_samples
0,0.50,0.515195,0.0,2073
1,0.60,0.515195,0.0,2073
2,0.70,0.515195,0.0,2073
3,0.80,0.515195,0.0,2073
4,0.90,0.515195,0.0,2073
5,0.95,0.515195,0.0,2073


## 16. Export to ONNX

In [20]:
onnx_path = Path(
    model.export(
        format="onnx",
        imgsz=IMAGE_SIZE,
        simplify=True,
    )
)

print("ONNX:", onnx_path)
print(f"Size: {onnx_path.stat().st_size / (1024 ** 2):.2f} MB")


Ultralytics 8.4.117 🚀 Python-3.14.4 torch-2.13.0+cu130 CPU (AMD Ryzen AI 9 365 w/ Radeon 880M)

PyTorch: starting from '/home/zubair/AI Development Directory/anomaly-detection/runs/classify/runs/classify/specialist_router_v2/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 18) (3.1 MB)

ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 0.2s, saved as '/home/zubair/AI Development Directory/anomaly-detection/runs/classify/runs/classify/specialist_router_v2/weights/best.onnx' (6.0 MB)

Export complete (0.2s)
Results saved to /home/zubair/AI Development Directory/anomaly-detection/runs/classify/runs/classify/specialist_router_v2/weights/best.onnx
Predict:         yolo predict task=classify model=/home/zubair/AI Development Directory/anomaly-detection/runs/classify/runs/classify/specialist_router_v2/weights/best.onnx imgsz=224 
Validate:        yolo val task=classify model=/home/zubair/AI Developme

## 17. Save routing metadata

In [24]:
# routing_accuracy = float(test_predictions["correct"].mean())

# metadata = {
#     "project": "VisionInspect",
#     "model_type": "specialist_router",
#     "architecture": MODEL_NAME,
#     "num_classes": len(ROUTER_CLASSES),
#     "classes": DOMAIN_TO_ID,
#     "mvtec_specialists": {
#         category: f"PatchCore({category})"
#         for category in MVTEC_CLASSES
#     },
#     "image_size": IMAGE_SIZE,
#     "seed": SEED,
#     "best_model": str(model.trainer.best),
#     "onnx_model": str(onnx_path),
#     "training_seconds": training_time,
#     "test_accuracy": routing_accuracy,
#     "wrong_specialist_rate": 1.0 - routing_accuracy,
# }

# metadata_path = ROUTER_ROOT / "router_metadata.json"
# metadata_path.write_text(
#     json.dumps(metadata, indent=2),
#     encoding="utf-8",
# )

# print(metadata_path)


# Expected deployment routing

```text
                         Input Image
                              │
                              ▼
                       YOLO26n Router
                              │
       ┌──────────────┬───────┴────────┬──────────────────────┐
       ▼              ▼                ▼                      ▼
     DAGM         Kolektor       Magnetic Tile          MVTec category
       │              │                │                      │
       ▼              ▼                ▼                      ▼
   YOLO-Seg       YOLO-Seg         YOLO-Seg             PatchCore
                                                               │
                          ┌──────────────┬─────────────────────┤
                          ▼              ▼                     ▼
                       bottle         cable                  ...
                          │              │                     │
                          ▼              ▼                     ▼
                    PatchCore      PatchCore             PatchCore
```

There are **18 routing classes / specialist destinations**: 3 supervised dataset specialists + 15 independent MVTec PatchCore specialists.

The router should be judged primarily by **per-specialist recall**, not just overall accuracy. A wrong route means the downstream specialist is inappropriate.
